# 01 - Explore the Risk Radar data

**Owner:** Daria  |  **Run:** before the hackathon, and again to check numbers with Alex

**What this data is:** every row is a forecast ACAPS made in the past about a humanitarian risk, with the outcome recorded afterwards. Status says where it stands: active (still being watched), materialised (it happened) or not materialised (the watching period ended without it happening).

This notebook checks the data against the numbers in the README and data dictionary, and collects the questions to ask Alex.

## 0. Setup

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 80)

# Find the repo folder (the one that contains src/risk_tools.py), wherever this notebook is run from
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "risk_tools.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError(f"Could not find src/risk_tools.py above {Path.cwd()}. Put risk_tools.py in the src/ folder of the repo.")
sys.path.append(str(ROOT / "src"))
from risk_tools import *
print("Repo folder:", ROOT)


Repo folder: c:\Users\iliev\Documents\github\HFG8-ACAPS-Ground-Truth


In [ ]:
DATA_PATH = ROOT / "data" / "risk-radar-export.csv"

## 1. Load and check the columns

In [ ]:
df = load_risks(DATA_PATH)
print("Missing expected columns:", check_columns(df) or "none")
n_countries = df["country"].nunique() if "country" in df else "n/a"
print(f"{len(df)} risks, {n_countries} distinct country values, {df.shape[1]} columns")
list(df.columns)

Missing expected columns: none
139 risks, 57 distinct country values, 25 columns


['risk_id',
 'risk_title',
 'risk_type',
 'iso',
 'country',
 'rationale',
 'source',
 'intensity_of_hazard',
 'impact',
 'probability',
 'risk_level',
 'risk_trend',
 'status',
 'what_materialised',
 'hazard_materialisation',
 'materialisation_sources',
 'triggers',
 'current_vulnerability_coping_capacities',
 'current_vulnerability_coping_capacities_sources',
 'expected_exposure',
 'expected_impact',
 'expected_impact_sources',
 'expected_priority_needs',
 'impact_materialisation_description',
 'rationale_update']

## 2. Status and rating

In [ ]:
df = add_outcome(df)
print(df["status"].value_counts(dropna=False), "\n")
print(df["risk_level"].value_counts(dropna=False))
pd.crosstab(df["risk_level"].fillna("not recorded"), df["status"].fillna("not recorded"))

status
Materialised        61
Not materialised    43
Active              28
<NA>                 7
Name: count, dtype: Int64 

risk_level
Medium    70
High      55
<NA>      11
Low        3
Name: count, dtype: Int64


status,Active,Materialised,Not materialised,not recorded
risk_level,,,,
High,17,32,6,0
Low,0,1,2,0
Medium,11,27,31,1
not recorded,0,1,4,6


## 3. Were the ratings right?
Among the risks where we know what happened, how often did each rating materialise?

In [ ]:
resolved = resolved_only(df)
rates = resolved.groupby("risk_level")["outcome"].agg(materialised="sum", resolved="count", rate="mean").round(2)
print(f"{len(resolved)} resolved risks")
rates

104 resolved risks


,materialised,resolved,rate
risk_level,,,
High,32,38,0.84
Low,1,3,0.33
Medium,27,58,0.47


In [ ]:
# Compare with the numbers in the data README (104 resolved, High 32 of 38, Medium 27 of 58)
expected = {"High": (32, 38), "Medium": (27, 58)}
print("Resolved:", len(resolved), "(README says 104)")
for lvl, (m, n) in expected.items():
    if lvl in rates.index:
        print(lvl, int(rates.loc[lvl, "materialised"]), "of", int(rates.loc[lvl, "resolved"]), f"(README says {m} of {n})")
print("Low has too few resolved risks to trust:", int(rates.loc["Low", "resolved"]) if "Low" in rates.index else "none")

Resolved: 104 (README says 104)
High 32 of 38 (README says 32 of 38)
Medium 27 of 58 (README says 27 of 58)
Low has too few resolved risks to trust: 3


## 4. How complete is each column?

In [ ]:
(df.notna().mean() * 100).round(0).sort_values().to_frame("filled %")

,filled %
impact_materialisation_description,35.0
hazard_materialisation,38.0
what_materialised,40.0
rationale_update,45.0
materialisation_sources,48.0
risk_trend,55.0
current_vulnerability_coping_capacities_sources,63.0
expected_impact_sources,64.0
expected_impact,67.0
expected_priority_needs,75.0


## 5. The time question
Scoring forecasts needs to know when a risk was published and when it resolved. There may not be a clean date column, so look for dates inside the update history.

In [ ]:
date_cols = [c for c in df.columns if "date" in c]
print("Columns with 'date' in the name:", date_cols or "none")
for _, r in df[df["rationale_update"].notna()].head(3).iterrows():
    print(r["risk_id"], "->", str(r["rationale_update"])[:300].replace("\n", " | "), "\n")

Columns with 'date' in the name: ['rationale_update']
 | Between April and early May 2025, the forced return of Afghan nationals from Pakistan intensified, aggravating humanitarian challenges in Afghanistan. In April alone, over 118,400 Afghans returned, including approximately 25,600 deportees. The majority of these returnees are undoc 

 | As of March 2026, Afghanistan faces escalating climate-driven risks linked to ongoing La Niña conditions that began in October 2025. Below-average snowfall and rainfall, combined with above-normal temperatures, are reinforcing drought conditions nationwide. Central, northeastern,  

 | Protection risks for communities in border areas increased in September as clashes and cross-border strikes between Pakistan and Afghanistan resumed after a temporary decline. Repeated Pakistani airstrikes and Afghan military operations have heightened the risk of civilian casualt 



In [ ]:
ds = df["rationale_update"].apply(date_summary)
df["first_date"] = ds.apply(lambda x: x[0])
df["last_date"] = ds.apply(lambda x: x[1])
df["n_dates"] = ds.apply(lambda x: x[2])
print("Risks with at least one date found:", int((df["n_dates"] > 0).sum()), "of", len(df))
df.loc[df["n_dates"] > 0, ["risk_id", "status", "first_date", "last_date", "n_dates"]].head(10)

c:\Users\iliev\Documents\github\HFG8-ACAPS-Ground-Truth\src\risk_tools.py:138: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  d = pd.to_datetime(m.replace("Sept", "Sep"), dayfirst=True, errors="coerce")


Risks with at least one date found: 63 of 139


,risk_id,status,first_date,last_date,n_dates
1,R0990,Materialised,2025-05-01,2025-05-08,2
2,R1015,Materialised,2025-02-09,2026-03-31,18
3,R1062,Active,0202-10-01,2026-10-07,4
4,R1067,Active,2026-09-02,2026-10-06,7
5,R1056,Materialised,2026-07-01,2026-09-07,11
8,R1039,Materialised,2026-05-01,2026-07-12,10
14,R1043,Active,2023-04-01,2026-09-30,21
15,R1066,Active,2026-05-19,2026-09-30,10
17,R1013,Not materialised,2024-11-10,2026-04-30,46
18,R1019,Materialised,2026-02-01,2026-06-11,18


## 6. Countries

In [ ]:
multi = df["country"].str.contains(",", na=False)
print("Risks covering several countries:", int(multi.sum()))
df["country"].str.split(",").explode().str.strip().value_counts().head(10)

Risks covering several countries: 7


country
Sudan                               13
Syria                                9
Democratic Republic of the Congo     6
Lebanon                              6
Mali                                 6
Afghanistan                          5
Somalia                              5
Niger                                4
Colombia                             4
Libya                                4
Name: count, dtype: int64

## 7. The demo risk, R1043

In [ ]:
demo = df[df["risk_id"] == "R1043"]
if demo.empty:
    print("R1043 not found")
else:
    for col, val in demo.iloc[0].items():
        text = str(val).replace("\n", " | ")
        print(f"{col}: {text[:200]}{'...' if len(text) > 200 else ''}")

risk_id: R1043
risk_title: Escalating cross-border insecurity linked to the Sudan conflict increases protection risks and humanitarian needs in eastern Chad
risk_type: Marked deterioration in an existing crisis
iso: TCD
country: Chad
rationale: Since early 2026, the conflict in Sudan has evolved from a humanitarian crisis affecting Chad into a direct security threat to eastern border areas. Repeated drone strikes, cross-border incursions, an...
 | https://globalinitiative.net/analysis/how-border-criminal-e...Risks-Instability.pdf
intensity_of_hazard: 4.0
impact: 4.0
probability: High
risk_level: High
risk_trend: Deteriorated
status: Active
what_materialised: nan
hazard_materialisation: nan
materialisation_sources: nan
 | :Additional drone strikes, shelling, or armed incursions near the Chad–Sudan border, particular...
current_vulnerability_coping_capacities: Before the Sudan conflict escalated, eastern Chad already faced high levels of poverty, food insecurity, weak infrastructure, and

## Questions for Alex (write the answers here)
1. Which direction matters most for the December pilot?
2. Which probability bands should we use? The infographic and the methodology PDF differ.
3. How do we tell when each risk was published and when it resolved? What exactly does "not materialised" mean?
4. How are older risks, which resolve over much longer periods, supposed to be treated?
5. Who sees what under each anonymity level, and is there a minimum number of answers before the group view is shown?